# Demo 01 : State schemas

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · typed-dict

We return a partial update. The other state keys keep their values.

In [2]:
import csv
from dataclasses import dataclass
from typing import TypedDict
from langgraph.graph import END, START, StateGraph

with open(data("demo-world/tickets.csv"), newline="", encoding="utf-8") as file:
    ticket = next(csv.DictReader(file))

class TicketState(TypedDict):
    ticket_id: str
    symptom: str
    status: str

def mark_open(state: TicketState):
    return {"status": "open"}

builder = StateGraph(TicketState)
builder.add_node("mark_open", mark_open)
builder.add_edge(START, "mark_open")
builder.add_edge("mark_open", END)
inputs = {"ticket_id": ticket["ticket_id"], "symptom": ticket["symptom"], "status": "new"}
print("typed dict :", builder.compile().invoke(inputs))

typed dict : {'ticket_id': 'BT-1001', 'symptom': 'Inverter communication dropout reported by drivers', 'status': 'open'}


We use a dataclass when state needs defaults. Nodes receive an instance and still return a dictionary update.

In [3]:
@dataclass
class TicketDefaults:
    ticket_id: str
    status: str = "new"

def inspect_default(state: TicketDefaults):
    print("default :", state.status)
    return {"status": "open"}

defaults = StateGraph(TicketDefaults)
defaults.add_node("open", inspect_default)
defaults.add_edge(START, "open")
defaults.add_edge("open", END)
print("dataclass :", defaults.compile().invoke({"ticket_id": ticket["ticket_id"]}))

default : new
dataclass : {'ticket_id': 'BT-1001', 'status': 'open'}


## b · pydantic

We validate the input to a node with Pydantic. We catch the invalid input so Run All continues.

In [4]:
from pydantic import BaseModel, Field, ValidationError

class ValidatedTicket(BaseModel):
    ticket_id: str
    hours: float = Field(ge=0)

def round_hours(state: ValidatedTicket):
    return {"hours": round(state.hours, 1)}

validated = StateGraph(ValidatedTicket)
validated.add_node("round", round_hours)
validated.add_edge(START, "round")
validated.add_edge("round", END)
validated_graph = validated.compile()
print("valid :", validated_graph.invoke({"ticket_id": ticket["ticket_id"], "hours": float(ticket["hours"])}))
try:
    validated_graph.invoke({"ticket_id": ticket["ticket_id"], "hours": -1})
except ValidationError as error:
    print("invalid :", error.errors()[0]["type"])

valid : {'ticket_id': 'BT-1001', 'hours': 2.5}
invalid : greater_than_equal


We return an invalid final update to locate the validation boundary. Pydantic validates node inputs, but the final dictionary returned by invoke is not automatically validated.

In [5]:
boundary = StateGraph(ValidatedTicket)
boundary.add_node("invalid_output", lambda state: {"hours": -1})
boundary.add_edge(START, "invalid_output")
boundary.add_edge("invalid_output", END)
print("final update :", boundary.compile().invoke({"ticket_id": ticket["ticket_id"], "hours": 1}))

final update : {'ticket_id': 'BT-1001', 'hours': -1}


## c · input-and-output

We expose the ticket as input and the status as output. Internal state can contain both.

In [6]:
class TicketInput(TypedDict):
    ticket_id: str

class StatusOutput(TypedDict):
    status: str

class InternalState(TypedDict):
    ticket_id: str
    status: str

public = StateGraph(InternalState, input_schema=TicketInput, output_schema=StatusOutput)
public.add_node("open", lambda state: {"status": "open"})
public.add_edge(START, "open")
public.add_edge("open", END)
print("public output :", public.compile().invoke({"ticket_id": ticket["ticket_id"]}))

public output : {'status': 'open'}


## d · private-state

We declare a private channel through a node's input schema. It is absent from invoke output, but remains visible in state streams.

In [7]:
class PrivateState(TypedDict):
    working_note: str

def prepare(state: TicketInput):
    return {"working_note": f"Inspect {state['ticket_id']}"}

def finish(state: PrivateState):
    return {"status": state["working_note"] + " at depot"}

private = StateGraph(InternalState, input_schema=TicketInput, output_schema=StatusOutput)
private.add_node("prepare", prepare)
private.add_node("finish", finish)
private.add_edge(START, "prepare")
private.add_edge("prepare", "finish")
private.add_edge("finish", END)
private_graph = private.compile()
print("output :", private_graph.invoke({"ticket_id": ticket["ticket_id"]}))
for part in private_graph.stream({"ticket_id": ticket["ticket_id"]}, stream_mode="values"):
    print("state :", part)

output : {'status': 'Inspect BT-1001 at depot'}
state : {'ticket_id': 'BT-1001'}
state : {'ticket_id': 'BT-1001', 'working_note': 'Inspect BT-1001'}
state : {'ticket_id': 'BT-1001', 'status': 'Inspect BT-1001 at depot', 'working_note': 'Inspect BT-1001'}


## Things to try

Optional : we can change a ticket, a thread id or a returned update and compare the saved state.